# Run SageMaker Clarify pre-training bias on a credit-scoring dataset and prove a balanced-class resample drops DPL below threshold

The credit-scoring team's most recent training set has a 4-to-1 imbalance favoring approved applications: 80% of approvals come from one gender facet, 20% from the other. Code review flagged it. The compliance lead wants documented pre-training bias metrics before the next training run and the SLA is absolute DPL below 0.1 on the protected attribute. You have one session to run the Clarify pre-training analysis on the raw set, capture the metrics, apply a SMOTE-style resample to balance the minority facet, prove the resample drops DPL under the threshold, and ship the analysis.json file compliance will sign off on.

Four artifacts ship in this lab:

- A 5000-row credit-scoring CSV with a `loan_approved` binary target, a `gender` facet column, and 7 numeric and categorical features, deliberately seeded with the 4-to-1 imbalance the scenario describes.
- A first SageMaker Clarify Processing Job that computes Class Imbalance (CI), Difference in Proportions of Labels (DPL), KL divergence, and Jensen-Shannon divergence on the raw set and writes analysis.json to S3.
- A SMOTE-style resampled CSV that equalizes the per-gender approval rate to within 5 percentage points.
- A second Clarify Processing Job on the resampled set that proves absolute DPL drops below 0.1.

**Time.** Plan for about 55 minutes hands-on. Each Clarify Processing Job runs roughly 5 minutes on ml.m5.large; the SMOTE resample runs locally in the kernel. Budget 90 minutes if the BiasConfig fights back.

**Cost.** This lab costs about two cents if both jobs land on the first try. SageMaker Processing on ml.m5.large is $0.115 per hour; two 5-minute runs is roughly $0.02. The Clarify container image is AWS-published and free. S3 storage for the two input CSVs and the analysis.json outputs is fractions of a penny. IAM is free. A realistic session with one Clarify config retry lands around $0.02 to $0.05. The cleanup cell shuts the bucket and role down so your bill stops the moment you finish.

This lab maps to AWS MLA-C01 Domain 1 (Data Preparation for ML, 28%) on detecting pre-training bias with SageMaker Clarify (CI, DPL, KL, JS metrics) and applying SMOTE-style resampling.

**Clarify vs Guardrails reminder.** SageMaker Clarify measures STATISTICAL fairness on tabular ML datasets and models. Amazon Bedrock Guardrails filters content safety for LLM prompts and responses. They solve different problems on different parts of the ML lifecycle, and mixing them up is the single most-cited trap question on the MLA-C01 exam. Lab 10 covers the Guardrails side; this lab is Clarify only.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus imbalanced-learn for the SMOTE step.
# Both are pinned to specific versions per LAB-CREATION-BLUEPRINT.md.

!pip install --quiet labrun-checks==0.6.0 imbalanced-learn==0.12.4

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. The bucket name fits S3's 63-character limit because the
# slug (47) plus a 12-digit account suffix lands at 59 characters.

import atexit
import getpass
import json
import time
from io import BytesIO

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "clarify-pretraining-bias-and-mitigation"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names. Bucket name is account-suffixed and resolved after STS returns.
CLARIFY_ROLE_NAME = f"labrun-{LAB_ID}-role"
INLINE_POLICY_NAME = "labrun-mla-lab04-clarify-policy"
CLARIFY_JOB_RAW = f"labrun-{LAB_ID}-raw"
CLARIFY_JOB_RESAMPLED = f"labrun-{LAB_ID}-resampled"

BUCKET_NAME = None
ACCOUNT_ID = None
CLARIFY_ROLE_ARN = None
CLARIFY_IMAGE_URI = None
RAW_DPL = None
RESAMPLED_DPL = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names. This cell must succeed before the manifest
# cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
CLARIFY_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{CLARIFY_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Clarify role ARN: {CLARIFY_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# Manifest is module-level and in reverse-creation order. The IAM role tears
# down before the bucket because nothing references it after the Processing
# Jobs finish; the bucket adapter wipes all objects under output/ and input/
# before deleting the bucket itself.
#
# SageMaker Processing Jobs cannot be deleted via API. Completed jobs expire
# automatically. The cleanup cell calls stop_processing_job on any in-flight
# job manually BEFORE run_cleanup walks the manifest; the manifest below
# contains only adapter-supported types.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=CLARIFY_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {CLARIFY_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Generate the 4-to-1 imbalanced credit-scoring CSV, stand up the bucket and role, upload to S3

Clarify needs three things to compute pre-training bias: a CSV in S3, an analysis_config.json describing the facet and label, and an IAM role with S3 read/write on the input and output prefixes plus a trust policy for the SageMaker service. Task 1 builds the dataset, the bucket, and the role.

Build it in this order:

1. Generate a 5000-row credit-scoring DataFrame with `numpy.random.default_rng(seed=42)` for determinism. Schema: `applicant_id` (int), `tenure_years` (float), `annual_income` (float), `loan_amount` (float), `credit_history_months` (int), `gender` (str in `["male", "female"]`, roughly 50/50), `employment_status` (str), `education_level` (str), `loan_approved` (int 0/1, the target). Seed the imbalance: P(approved | male) approximately 0.8, P(approved | female) approximately 0.2. That is the 4-to-1 the scenario describes.
2. Create the S3 bucket `labrun-clarify-pretraining-bias-and-mitigation-{your-account-id}` and tag it `labrun:lab-slug=clarify-pretraining-bias-and-mitigation`.
3. Upload the CSV to `s3://{bucket}/input/raw.csv`.
4. Create the IAM role `labrun-clarify-pretraining-bias-and-mitigation-role` with a trust policy for `sagemaker.amazonaws.com` and an inline policy granting S3 read/write on the bucket plus `s3:ListBucket` on the bucket itself.
5. Sleep 10 seconds so IAM propagation lands before Task 2 calls `create_processing_job`.

Tagging at creation matters. The orphan scan in the cleanup cell looks for the lab tag; an untagged bucket is invisible to the tag audit.

In [ ]:
# NBVAL_SKIP
# Task 1: Generate the imbalanced credit-scoring CSV, create the bucket and
# the IAM role, upload the CSV. The data generator is deterministic so the
# checkpoint can assert on the approval rates without flapping.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def generate_credit_df(n_rows: int = 5000) -> pd.DataFrame:
    rng = np.random.default_rng(seed=42)
    gender = rng.choice(["male", "female"], size=n_rows, p=[0.5, 0.5])
    employment_status = rng.choice(
        ["employed", "self_employed", "unemployed"], size=n_rows, p=[0.7, 0.2, 0.1]
    )
    education_level = rng.choice(
        ["high_school", "bachelors", "graduate"], size=n_rows, p=[0.4, 0.4, 0.2]
    )
    tenure_years = rng.normal(loc=6.0, scale=3.0, size=n_rows).clip(0, 40).round(2)
    annual_income = rng.normal(loc=70000, scale=25000, size=n_rows).clip(15000, 250000).round(2)
    loan_amount = rng.normal(loc=20000, scale=8000, size=n_rows).clip(2000, 80000).round(2)
    credit_history_months = rng.integers(low=6, high=300, size=n_rows)

    # Seed the 4-to-1 imbalance: P(approved | male) ~= 0.8, P(approved | female) ~= 0.2.
    approved = np.zeros(n_rows, dtype=int)
    male_mask = gender == "male"
    female_mask = gender == "female"
    approved[male_mask] = rng.binomial(n=1, p=0.8, size=male_mask.sum())
    approved[female_mask] = rng.binomial(n=1, p=0.2, size=female_mask.sum())

    return pd.DataFrame({
        "applicant_id": np.arange(1, n_rows + 1),
        "tenure_years": tenure_years,
        "annual_income": annual_income,
        "loan_amount": loan_amount,
        "credit_history_months": credit_history_months,
        "gender": gender,
        "employment_status": employment_status,
        "education_level": education_level,
        "loan_approved": approved,
    })


raw_df = generate_credit_df()
print("Generated dataframe:")
print(f"  rows: {len(raw_df)}")
print(f"  P(approved | male):   {raw_df[raw_df.gender == 'male'].loan_approved.mean():.3f}")
print(f"  P(approved | female): {raw_df[raw_df.gender == 'female'].loan_approved.mean():.3f}")

# YOUR CODE: Create the bucket with s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects CreateBucketConfiguration; other regions require it.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# YOUR CODE: Serialize raw_df to CSV bytes and upload to s3://{BUCKET_NAME}/input/raw.csv
# using s3.put_object(Bucket=BUCKET_NAME, Key="input/raw.csv", Body=<csv_bytes>).
# Hint: raw_df.to_csv(index=False).encode("utf-8") gives you the bytes.
print(f"Uploaded s3://{BUCKET_NAME}/input/raw.csv")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    # YOUR CODE: Create the role using iam.create_role() with
    # RoleName=CLARIFY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
    # Description="labrun mla lab 04 clarify role", and
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created role: {CLARIFY_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {CLARIFY_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BucketRW",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Sid": "BucketList",
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
    ],
}

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=CLARIFY_ROLE_NAME, PolicyName=INLINE_POLICY_NAME,
# PolicyDocument=json.dumps(inline_policy).
print(f"Attached inline policy {INLINE_POLICY_NAME} to {CLARIFY_ROLE_NAME}")

# IAM propagation. create_processing_job rejects roles that have not finished
# propagating; give it 10 seconds.
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)
print("Done.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: the raw CSV is in S3 with the right row count and the right
# 4-to-1 facet imbalance. Validator reads the first 1 MB of the object via
# Range, parses it with pandas, and asserts on the per-gender approval rate.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            s3_client.head_object(Bucket=BUCKET_NAME, Key="input/raw.csv")
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str in ("404", "NoSuchKey", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"s3://{BUCKET_NAME}/input/raw.csv was not found. Upload the "
                        f"generated CSV before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key="input/raw.csv")
            body_bytes = obj["Body"].read()
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            df = pd.read_csv(BytesIO(body_bytes))
        except Exception as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"raw.csv could not be parsed by pandas: {e}",
            )

        if "gender" not in df.columns or "loan_approved" not in df.columns:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"raw.csv is missing required columns. Need 'gender' and "
                    f"'loan_approved'. Found: {list(df.columns)}"
                ),
            )

        row_count = len(df)
        if not (4900 <= row_count <= 5100):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"raw.csv has {row_count} rows. Expected roughly 5000. "
                    f"Re-run the generator with n_rows=5000."
                ),
            )

        rates = df.groupby("gender")["loan_approved"].mean().to_dict()
        p_male = rates.get("male")
        p_female = rates.get("female")
        if p_male is None or p_female is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"raw.csv 'gender' column must contain both 'male' and 'female'. "
                    f"Found rates: {rates}"
                ),
            )
        if not (p_male > 0.6 and p_female < 0.4):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"raw.csv does not have the expected 4-to-1 facet imbalance. "
                    f"P(approved | male) = {p_male:.3f} (need > 0.6); "
                    f"P(approved | female) = {p_female:.3f} (need < 0.4)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Three blanks in the cell: create the bucket, upload the CSV, create the role, attach the inline policy. The checkpoint message tells you whether the object is missing, the row count is wrong, or the facet imbalance does not land.

</details>

<details><summary>Hint 2 (direction)</summary>

In `us-east-1` use plain `s3.create_bucket(Bucket=BUCKET_NAME)` with no `CreateBucketConfiguration`. CSV bytes come from `raw_df.to_csv(index=False).encode("utf-8")`. The role is created with `iam.create_role(RoleName=..., AssumeRolePolicyDocument=json.dumps(trust_policy), Tags=[...])` and the inline policy is attached with `iam.put_role_policy(RoleName=..., PolicyName=..., PolicyDocument=json.dumps(inline_policy))`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the four lines you need to fill in):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

csv_bytes = raw_df.to_csv(index=False).encode("utf-8")
s3.put_object(Bucket=BUCKET_NAME, Key="input/raw.csv", Body=csv_bytes)

iam.create_role(
    RoleName=CLARIFY_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="labrun mla lab 04 clarify role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.put_role_policy(
    RoleName=CLARIFY_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)
```

</details>

**Wallet check.** One S3 bucket, one CSV upload at roughly 400 KB, an IAM role, and an inline policy. S3 control-plane plus a single put_object sits inside the always-free tier. IAM and STS are free. Damage so far: about $0.00. Your coffee this morning cost infinitely more.

## Task 2: Run the Clarify Processing Job on the raw CSV and capture the DPL

Clarify is a SageMaker Processing Job that takes a CSV plus an `analysis_config.json` and writes an `analysis.json` to S3 with the computed bias metrics. The job runs in an AWS-published container; you pass the container URI, the role ARN, the inputs, and the outputs, and `create_processing_job` returns immediately. The job runs roughly 5 minutes on ml.m5.large.

Build it in this order:

1. Fetch the Clarify container URI via `sagemaker.image_uris.retrieve(framework="clarify", region="us-east-1", version="1.0")`. The URI is region-specific; hardcoding a stale URI causes ImagePullBackoff. The `sagemaker` import below is the SageMaker Python SDK, which the SageMaker Processing container fleet uses for image lookups.
2. Upload an `analysis_config.json` to `s3://{bucket}/input/analysis_config.json` describing the dataset, the label, and the facet. The exam-relevant fields are `dataset_type="text/csv"`, `label="loan_approved"`, `label_values_or_threshold=[1]` (1 is the favored value), `facet=[{"name_or_index": "gender", "value_or_threshold": ["female"]}]`, and `methods={"pre_training_bias": {"methods": "all"}}`. The `"all"` shortcut tells Clarify to compute every pre-training metric: CI, DPL, KL, JS, plus several others.
3. Call `sm.create_processing_job` with ProcessingInputs pointing at the bucket prefix containing `raw.csv` AND the `analysis_config.json` (Clarify expects both at the input mount), and ProcessingOutputs writing to `s3://{bucket}/output/raw/`.
4. Poll `sm.describe_processing_job` every 15 seconds for up to 15 minutes until `ProcessingJobStatus` is `Completed`.
5. Read `analysis.json` from `output/raw/`, parse the `pre_training_bias_metrics`, and print the DPL value.

Why `label_values_or_threshold=[1]`? Clarify needs to know which target value is the favored outcome. In this scenario, `loan_approved=1` is favored; `loan_approved=0` is unfavored. A misconfigured `label_values_or_threshold` produces a DPL with the wrong sign or a metric on the wrong direction.

In [ ]:
# NBVAL_SKIP
# Task 2: Submit the first Clarify Processing Job on the raw CSV.
# The Clarify container reads input/raw.csv and input/analysis_config.json
# from the input mount, computes pre-training bias metrics, and writes
# analysis.json to output/raw/.

import sagemaker

sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Fetch the Clarify container URI for this region using
# sagemaker.image_uris.retrieve(framework="clarify", region=REGION, version="1.0").
# Assign the result to CLARIFY_IMAGE_URI.
print(f"Clarify image URI: {CLARIFY_IMAGE_URI}")

analysis_config_raw = {
    "dataset_type": "text/csv",
    "headers": [
        "applicant_id",
        "tenure_years",
        "annual_income",
        "loan_amount",
        "credit_history_months",
        "gender",
        "employment_status",
        "education_level",
        "loan_approved",
    ],
    "label": "loan_approved",
    "label_values_or_threshold": [1],
    "facet": [
        {"name_or_index": "gender", "value_or_threshold": ["female"]},
    ],
    "methods": {
        "pre_training_bias": {"methods": "all"},
        "report": {"name": "report", "title": "Clarify pre-training bias report"},
    },
}

s3.put_object(
    Bucket=BUCKET_NAME,
    Key="input/analysis_config.json",
    Body=json.dumps(analysis_config_raw).encode("utf-8"),
)
print(f"Uploaded s3://{BUCKET_NAME}/input/analysis_config.json")

processing_inputs = [
    {
        "InputName": "dataset",
        "S3Input": {
            "S3Uri": f"s3://{BUCKET_NAME}/input/raw.csv",
            "LocalPath": "/opt/ml/processing/input/data",
            "S3DataType": "S3Prefix",
            "S3InputMode": "File",
            "S3CompressionType": "None",
        },
    },
    {
        "InputName": "analysis_config",
        "S3Input": {
            "S3Uri": f"s3://{BUCKET_NAME}/input/analysis_config.json",
            "LocalPath": "/opt/ml/processing/input/config",
            "S3DataType": "S3Prefix",
            "S3InputMode": "File",
            "S3CompressionType": "None",
        },
    },
]
processing_outputs = {
    "Outputs": [
        {
            "OutputName": "analysis_result",
            "S3Output": {
                "S3Uri": f"s3://{BUCKET_NAME}/output/raw/",
                "LocalPath": "/opt/ml/processing/output",
                "S3UploadMode": "EndOfJob",
            },
        }
    ]
}

# YOUR CODE: Submit the Processing Job using sm.create_processing_job() with
# ProcessingJobName=CLARIFY_JOB_RAW,
# ProcessingInputs=processing_inputs,
# ProcessingOutputConfig=processing_outputs,
# RoleArn=CLARIFY_ROLE_ARN,
# AppSpecification={"ImageUri": CLARIFY_IMAGE_URI},
# ProcessingResources={"ClusterConfig": {"InstanceCount": 1, "InstanceType": "ml.m5.large", "VolumeSizeInGB": 20}},
# StoppingCondition={"MaxRuntimeInSeconds": 900},
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Submitted Processing Job: {CLARIFY_JOB_RAW}")

# Poll until Completed. Clarify runs roughly 5 minutes on ml.m5.large;
# the ceiling is 15 minutes.
print("Clarify is reading your dataset, give it about 5 minutes...")
elapsed = 0
status = "InProgress"
while elapsed < 900:
    resp = sm.describe_processing_job(ProcessingJobName=CLARIFY_JOB_RAW)
    status = resp["ProcessingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Processing Job ended with status {status}. Inspect the job in the SageMaker console.")
    raise SystemExit(1)

print(f"Processing Job {CLARIFY_JOB_RAW} completed in roughly {elapsed}s.")

# Read analysis.json from output/raw/ and parse the DPL.
analysis_obj = s3.get_object(Bucket=BUCKET_NAME, Key="output/raw/analysis.json")
analysis_raw = json.loads(analysis_obj["Body"].read())

def _extract_dpl(analysis: dict) -> float | None:
    metrics_root = analysis.get("pre_training_bias_metrics", {})
    facets = metrics_root.get("facets", {})
    gender_facets = facets.get("gender", [])
    if not gender_facets:
        return None
    for metric in gender_facets[0].get("metrics", []):
        if metric.get("name") == "DPL":
            return metric.get("value")
    return None


RAW_DPL = _extract_dpl(analysis_raw)
print(f"Raw DPL on gender facet: {RAW_DPL}")
if RAW_DPL is None:
    print("Could not find DPL in analysis.json. Inspect the file structure manually.")
else:
    print(f"|DPL| = {abs(RAW_DPL):.3f}. Compliance threshold is 0.1.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: the Processing Job is Completed AND analysis.json contains a
# DPL whose absolute value exceeds 0.1 (above the compliance threshold). The
# validator re-queries describe_processing_job and re-reads analysis.json
# from S3 instead of trusting notebook local variables.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_processing_job(ProcessingJobName=CLARIFY_JOB_RAW)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Processing Job {CLARIFY_JOB_RAW} was not found. Submit the "
                        f"job before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["ProcessingJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Processing Job {CLARIFY_JOB_RAW} status is "
                    f"{job['ProcessingJobStatus']}, not Completed. Wait for the poll "
                    f"loop in the task cell to finish."
                ),
            )

        try:
            analysis_obj = s3_client.get_object(
                Bucket=BUCKET_NAME, Key="output/raw/analysis.json"
            )
            analysis = json.loads(analysis_obj["Body"].read())
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"output/raw/analysis.json was not written. The Processing Job "
                        f"reported Completed but the analysis output is missing. Check "
                        f"the job logs in CloudWatch."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        # Defensive walk in case of minor schema variants across Clarify versions.
        metrics_root = analysis.get("pre_training_bias_metrics", {})
        facets = metrics_root.get("facets", {})
        gender_facets = facets.get("gender", [])
        if not gender_facets:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"analysis.json has no pre_training_bias_metrics.facets.gender entry. "
                    f"Check that the BiasConfig facet name_or_index is 'gender'."
                ),
            )
        dpl_value = None
        for metric in gender_facets[0].get("metrics", []):
            if metric.get("name") == "DPL":
                dpl_value = metric.get("value")
                break
        if dpl_value is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"analysis.json has no DPL metric under the gender facet. "
                    f"Confirm methods.pre_training_bias.methods is 'all' in the config."
                ),
            )
        if abs(dpl_value) <= 0.1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"|DPL| = {abs(dpl_value):.3f} on the raw dataset, expected > 0.1. "
                    f"The 4-to-1 imbalance in raw.csv should produce |DPL| near 0.6. "
                    f"Confirm raw.csv has the seeded imbalance (Checkpoint 1)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: the image URI lookup and the `create_processing_job` call. The checkpoint message tells you whether the job is missing, still running, or whether analysis.json reports a DPL under threshold (which means the raw set was not as imbalanced as expected).

</details>

<details><summary>Hint 2 (direction)</summary>

`sagemaker.image_uris.retrieve(framework="clarify", region=REGION, version="1.0")` returns the Clarify image URI as a string. `sm.create_processing_job(...)` takes the kwargs already listed in the comment. ml.m5.large at $0.115/hour is the right instance type; `VolumeSizeInGB=20` is plenty for a 400 KB CSV.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the two lines you need to fill in):

```python
CLARIFY_IMAGE_URI = sagemaker.image_uris.retrieve(
    framework="clarify", region=REGION, version="1.0"
)

sm.create_processing_job(
    ProcessingJobName=CLARIFY_JOB_RAW,
    ProcessingInputs=processing_inputs,
    ProcessingOutputConfig=processing_outputs,
    RoleArn=CLARIFY_ROLE_ARN,
    AppSpecification={"ImageUri": CLARIFY_IMAGE_URI},
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.m5.large",
            "VolumeSizeInGB": 20,
        }
    },
    StoppingCondition={"MaxRuntimeInSeconds": 900},
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** One SageMaker Processing Job at ml.m5.large ($0.115/hour) for roughly 5 minutes lands at about a penny. The Clarify container image is AWS-published and free. Damage so far: about $0.01. Your coffee this morning still cost more.

## Task 3: Apply a SMOTE-style resample to balance approval rates across the gender facet

DPL is above threshold. The compliance fix is to balance the per-facet approval rate: bring P(approved | female) up so it lands within 5 percentage points of P(approved | male). The standard tool is `imbalanced-learn`, which ships SMOTE (Synthetic Minority Oversampling Technique) and a simpler `RandomOverSampler` that copies minority rows.

SMOTE generates synthetic minority-class rows by interpolating between nearest neighbors in feature space. The catch: SMOTE's k-nearest-neighbor logic needs numeric features. The dataset has three string columns (`gender`, `employment_status`, `education_level`). The workaround is to encode strings to integers with `pd.factorize` before SMOTE, then decode back to strings after. If SMOTE still raises on the small dataset, fall back to `RandomOverSampler` which works on any feature type.

Resample strategy for this lab: split the dataframe by gender, oversample the minority-approved group within each gender so within-gender approval lands near 0.5, then concatenate. The SMOTE call only operates on rows where balancing is needed, not on the whole frame at once.

Build it in this order:

1. Download `input/raw.csv` from S3 and load it into a fresh DataFrame.
2. Split by gender. For each gender subset, treat `loan_approved` as the y; use `imbalanced-learn`'s `SMOTE` (or `RandomOverSampler` if SMOTE raises) with `sampling_strategy="auto"` to oversample the minority class within that subset.
3. Concatenate the resampled subsets, decode any factorized columns back to strings, save to `input/resampled.csv`, upload to S3.
4. Verify locally that P(approved | female) is now within 5 percentage points of P(approved | male).

In [ ]:
# NBVAL_SKIP
# Task 3: SMOTE-style resample. Balance the per-gender approval rate by
# oversampling the minority-approved class within each gender subset.

from imblearn.over_sampling import SMOTE, RandomOverSampler

raw_obj = s3.get_object(Bucket=BUCKET_NAME, Key="input/raw.csv")
raw_csv_bytes = raw_obj["Body"].read()
raw_loaded = pd.read_csv(BytesIO(raw_csv_bytes))
print(f"Loaded raw.csv from S3, rows: {len(raw_loaded)}")

string_cols = ["gender", "employment_status", "education_level"]
factorizations: dict[str, np.ndarray] = {}
encoded = raw_loaded.copy()
for col in string_cols:
    codes, uniques = pd.factorize(encoded[col])
    encoded[col] = codes
    factorizations[col] = uniques


def resample_subset(subset: pd.DataFrame) -> pd.DataFrame:
    y = subset["loan_approved"].values
    X = subset.drop(columns=["loan_approved"]).values
    # If the subset is already balanced or the minority class has fewer than
    # k_neighbors+1 rows, SMOTE will raise. Fall back to RandomOverSampler.
    try:
        # YOUR CODE: Instantiate SMOTE with sampling_strategy="auto" and
        # random_state=42, then call .fit_resample(X, y) to get X_res, y_res.
        X_res, y_res = X, y  # placeholder; replace with the SMOTE call
        raise NotImplementedError("replace this stub with your SMOTE call above")
    except Exception as e:
        print(f"  SMOTE fell back to RandomOverSampler ({type(e).__name__}: {e})")
        ros = RandomOverSampler(sampling_strategy="auto", random_state=42)
        X_res, y_res = ros.fit_resample(X, y)
    res_df = pd.DataFrame(X_res, columns=subset.drop(columns=["loan_approved"]).columns)
    res_df["loan_approved"] = y_res
    return res_df


resampled_parts = []
for gender_code in encoded["gender"].unique():
    subset = encoded[encoded["gender"] == gender_code].copy()
    print(f"Resampling gender_code={gender_code} ({factorizations['gender'][gender_code]!r}), rows={len(subset)}")
    resampled = resample_subset(subset)
    resampled_parts.append(resampled)

resampled_encoded = pd.concat(resampled_parts, ignore_index=True)

# Decode factorized columns back to strings before upload.
for col, uniques in factorizations.items():
    resampled_encoded[col] = resampled_encoded[col].astype(int).map(lambda i: uniques[i])

# Reorder columns to match the original schema.
resampled_df = resampled_encoded[raw_loaded.columns]

rates_resampled = resampled_df.groupby("gender")["loan_approved"].mean().to_dict()
print()
print(f"Resampled rows: {len(resampled_df)}")
print(f"  P(approved | male):   {rates_resampled.get('male', float('nan')):.3f}")
print(f"  P(approved | female): {rates_resampled.get('female', float('nan')):.3f}")

resampled_bytes = resampled_df.to_csv(index=False).encode("utf-8")
s3.put_object(Bucket=BUCKET_NAME, Key="input/resampled.csv", Body=resampled_bytes)
print(f"Uploaded s3://{BUCKET_NAME}/input/resampled.csv")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: resampled.csv exists in S3 and per-gender approval rates are
# within 5 percentage points of each other.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            s3_client.head_object(Bucket=BUCKET_NAME, Key="input/resampled.csv")
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str in ("404", "NoSuchKey"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"s3://{BUCKET_NAME}/input/resampled.csv was not found. Upload the "
                        f"resampled CSV before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        obj = s3_client.get_object(Bucket=BUCKET_NAME, Key="input/resampled.csv")
        df = pd.read_csv(BytesIO(obj["Body"].read()))

        if "gender" not in df.columns or "loan_approved" not in df.columns:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"resampled.csv is missing required columns. Need 'gender' and "
                    f"'loan_approved'. Found: {list(df.columns)}"
                ),
            )

        rates = df.groupby("gender")["loan_approved"].mean().to_dict()
        p_male = rates.get("male")
        p_female = rates.get("female")
        if p_male is None or p_female is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"resampled.csv 'gender' column must contain both 'male' and 'female'. "
                    f"Found rates: {rates}"
                ),
            )
        if abs(p_male - p_female) > 0.05:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Per-gender approval rates are not within 5 percentage points: "
                    f"P(approved | male) = {p_male:.3f}, P(approved | female) = {p_female:.3f}. "
                    f"Re-run SMOTE with sampling_strategy='auto' or fall back to "
                    f"RandomOverSampler."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

One blank inside `resample_subset`: instantiate `SMOTE` and call `.fit_resample(X, y)`. The fallback to `RandomOverSampler` is already wired in the `except` block; SMOTE is the preferred path because it generates synthetic rows rather than duplicating existing ones.

</details>

<details><summary>Hint 2 (direction)</summary>

`from imblearn.over_sampling import SMOTE` is already imported. The constructor takes `sampling_strategy="auto"` (which oversamples the minority class until it matches the majority) and `random_state=42` for determinism. `.fit_resample(X, y)` returns `X_res, y_res` as numpy arrays.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the one line you need to fill in inside `resample_subset`):

```python
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_res, y_res = smote.fit_resample(X, y)
```

Remove the `raise NotImplementedError(...)` line immediately below it so the SMOTE result is actually returned. The fallback to `RandomOverSampler` only fires if SMOTE raises (typically when the minority class has fewer rows than `k_neighbors + 1`, which can happen on tiny per-facet subsets).

</details>

**Wallet check.** SMOTE runs entirely in the notebook kernel. No SageMaker call, no Processing Job, no compute charge. The resampled CSV upload is a fraction of a penny. Damage so far: about $0.01. Your coffee still wins.


## Task 4: Re-run Clarify on the resampled CSV and prove DPL drops below threshold

The resampled CSV is balanced. The compliance team needs the analysis.json artifact that documents the drop. Submit a second Clarify Processing Job against `input/resampled.csv`, write to `output/resampled/`, parse the new DPL, print the before/after delta.

Build it in this order:

1. Upload a second `analysis_config.json` (call it `analysis_config_resampled.json`) that points at `input/resampled.csv` instead of `input/raw.csv`. The BiasConfig is identical to Task 2's config; only the input dataset changes.
2. Submit a second Processing Job named `labrun-clarify-pretraining-bias-and-mitigation-resampled` with ProcessingInputs pointing at the resampled CSV plus the new config, and ProcessingOutputs writing to `output/resampled/`.
3. Poll until `Completed`.
4. Read `output/resampled/analysis.json`, parse the new DPL, and print the before/after numbers.

In [ ]:
# NBVAL_SKIP
# Task 4: Submit the second Clarify Processing Job on the resampled CSV.
# Same BiasConfig, different input dataset, different output prefix.

analysis_config_resampled = dict(analysis_config_raw)  # shallow copy is fine

s3.put_object(
    Bucket=BUCKET_NAME,
    Key="input/analysis_config_resampled.json",
    Body=json.dumps(analysis_config_resampled).encode("utf-8"),
)
print(f"Uploaded s3://{BUCKET_NAME}/input/analysis_config_resampled.json")

processing_inputs_resampled = [
    {
        "InputName": "dataset",
        "S3Input": {
            "S3Uri": f"s3://{BUCKET_NAME}/input/resampled.csv",
            "LocalPath": "/opt/ml/processing/input/data",
            "S3DataType": "S3Prefix",
            "S3InputMode": "File",
            "S3CompressionType": "None",
        },
    },
    {
        "InputName": "analysis_config",
        "S3Input": {
            "S3Uri": f"s3://{BUCKET_NAME}/input/analysis_config_resampled.json",
            "LocalPath": "/opt/ml/processing/input/config",
            "S3DataType": "S3Prefix",
            "S3InputMode": "File",
            "S3CompressionType": "None",
        },
    },
]
processing_outputs_resampled = {
    "Outputs": [
        {
            "OutputName": "analysis_result",
            "S3Output": {
                "S3Uri": f"s3://{BUCKET_NAME}/output/resampled/",
                "LocalPath": "/opt/ml/processing/output",
                "S3UploadMode": "EndOfJob",
            },
        }
    ]
}

# YOUR CODE: Submit the second Processing Job using sm.create_processing_job()
# with ProcessingJobName=CLARIFY_JOB_RESAMPLED, ProcessingInputs=
# processing_inputs_resampled, ProcessingOutputConfig=processing_outputs_resampled,
# RoleArn=CLARIFY_ROLE_ARN, AppSpecification={"ImageUri": CLARIFY_IMAGE_URI},
# ProcessingResources={"ClusterConfig": {"InstanceCount": 1, "InstanceType":
# "ml.m5.large", "VolumeSizeInGB": 20}}, StoppingCondition=
# {"MaxRuntimeInSeconds": 900}, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Submitted Processing Job: {CLARIFY_JOB_RESAMPLED}")

print("Clarify is reading the resampled dataset, give it about 5 more minutes...")
elapsed = 0
status = "InProgress"
while elapsed < 900:
    resp = sm.describe_processing_job(ProcessingJobName=CLARIFY_JOB_RESAMPLED)
    status = resp["ProcessingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Processing Job ended with status {status}. Inspect the job in the SageMaker console.")
    raise SystemExit(1)

print(f"Processing Job {CLARIFY_JOB_RESAMPLED} completed in roughly {elapsed}s.")

analysis_obj_r = s3.get_object(Bucket=BUCKET_NAME, Key="output/resampled/analysis.json")
analysis_resampled = json.loads(analysis_obj_r["Body"].read())
RESAMPLED_DPL = _extract_dpl(analysis_resampled)
print()
print(f"Before (raw):       DPL = {RAW_DPL}")
print(f"After  (resampled): DPL = {RESAMPLED_DPL}")
if RESAMPLED_DPL is not None:
    print(f"|DPL| dropped from {abs(RAW_DPL):.3f} to {abs(RESAMPLED_DPL):.3f}. Compliance threshold is 0.1.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: the second Processing Job is Completed AND analysis.json on
# the resampled dataset reports |DPL| < 0.1 (below the compliance threshold).

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_processing_job(ProcessingJobName=CLARIFY_JOB_RESAMPLED)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Processing Job {CLARIFY_JOB_RESAMPLED} was not found. "
                        f"Submit the second job before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["ProcessingJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Processing Job {CLARIFY_JOB_RESAMPLED} status is "
                    f"{job['ProcessingJobStatus']}, not Completed. Wait for the poll "
                    f"loop in the task cell to finish."
                ),
            )

        try:
            analysis_obj = s3_client.get_object(
                Bucket=BUCKET_NAME, Key="output/resampled/analysis.json"
            )
            analysis = json.loads(analysis_obj["Body"].read())
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"output/resampled/analysis.json was not written. The job reported "
                        f"Completed but the analysis output is missing. Check the job logs "
                        f"in CloudWatch."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        metrics_root = analysis.get("pre_training_bias_metrics", {})
        facets = metrics_root.get("facets", {})
        gender_facets = facets.get("gender", [])
        if not gender_facets:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"analysis.json has no pre_training_bias_metrics.facets.gender entry."
                ),
            )
        dpl_value = None
        for metric in gender_facets[0].get("metrics", []):
            if metric.get("name") == "DPL":
                dpl_value = metric.get("value")
                break
        if dpl_value is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"analysis.json has no DPL metric under the gender facet."
                ),
            )
        if abs(dpl_value) >= 0.1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"|DPL| on the resampled dataset = {abs(dpl_value):.3f}, still at or "
                    f"above the 0.1 compliance threshold. Tighten the resample so per-gender "
                    f"approval rates land within 5 percentage points of each other "
                    f"(Checkpoint 3), then re-run the second Processing Job."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Only one blank: a second `create_processing_job` call. The BiasConfig in `analysis_config_resampled` already points at the right input via the new ProcessingInputs list. If the checkpoint reports |DPL| still at or above 0.1, the resampled CSV is not balanced enough (go back and re-check Task 3).

</details>

<details><summary>Hint 2 (direction)</summary>

The shape of `create_processing_job` is identical to Task 2 with three substitutions: `ProcessingJobName=CLARIFY_JOB_RESAMPLED`, `ProcessingInputs=processing_inputs_resampled`, and `ProcessingOutputConfig=processing_outputs_resampled`. Everything else (`RoleArn`, `AppSpecification`, `ProcessingResources`, `StoppingCondition`, `Tags`) stays the same.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the one block you need to fill in):

```python
sm.create_processing_job(
    ProcessingJobName=CLARIFY_JOB_RESAMPLED,
    ProcessingInputs=processing_inputs_resampled,
    ProcessingOutputConfig=processing_outputs_resampled,
    RoleArn=CLARIFY_ROLE_ARN,
    AppSpecification={"ImageUri": CLARIFY_IMAGE_URI},
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.m5.large",
            "VolumeSizeInGB": 20,
        }
    },
    StoppingCondition={"MaxRuntimeInSeconds": 900},
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Second Clarify Processing Job at ml.m5.large for roughly 5 minutes lands at another penny. Damage so far: about $0.02. Your coffee this morning cost an order of magnitude more.

## Cleanup

Two Processing Jobs ran to completion; they cannot be deleted via API and bill nothing while idle, so the cleanup cell only needs to deal with any in-flight job (defensive `stop_processing_job` calls), the IAM role, and the bucket. The `s3_bucket` adapter wipes every object under `input/` and `output/` before deleting the bucket itself; the `iam_role` adapter deletes the inline policy first and then the role.

Re-running the cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Defensive stop on any still-running Processing Job, then
# run_cleanup walks the manifest (iam_role -> s3_bucket).

import sys

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings: list[str] = []

for job_name in (CLARIFY_JOB_RAW, CLARIFY_JOB_RESAMPLED):
    try:
        resp = sm_cleanup.describe_processing_job(ProcessingJobName=job_name)
        if resp["ProcessingJobStatus"] == "InProgress":
            try:
                sm_cleanup.stop_processing_job(ProcessingJobName=job_name)
                print(f"Requested stop on in-flight Processing Job {job_name}")
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ResourceNotFound",
                    "ValidationException",
                ):
                    manual_warnings.append(
                        f"FAILED TO STOP: processing_job {job_name!r}. Error: {e}. "
                        f"Run manually: aws sagemaker stop-processing-job "
                        f"--processing-job-name {job_name}"
                    )
        else:
            print(f"Processing Job {job_name} is {resp['ProcessingJobStatus']}, no stop needed.")
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ResourceNotFound", "ValidationException"):
            manual_warnings.append(
                f"FAILED TO DESCRIBE: processing_job {job_name!r}. Error: {e}."
            )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0  # this lab has no hourly-billed critical resources
critical_destroyed = 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean") else "dirty"
cleanup(status=final_status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.02.** Two Clarify Processing Jobs on ml.m5.large at $0.115/hour for roughly 5 minutes each comes out to two pennies. The Clarify container image is AWS-published and free. S3 storage for the two input CSVs and the analysis.json outputs is fractions of a penny. IAM and STS are free. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the four pre-training bias metrics Clarify computes: Class Imbalance (CI), Difference in Proportions of Labels (DPL), Kullback-Leibler (KL) divergence, and Jensen-Shannon (JS) divergence. For each one, write one sentence on what it measures, then write one sentence on when you would prefer that metric over the others. Why does the compliance threshold in the scenario use DPL specifically and not, say, KL? What is the practical thing DPL captures that the divergence metrics do not?

2. Explain the difference between SageMaker Clarify and Amazon Bedrock Guardrails. Both relate to safety in some sense but they solve different problems on different parts of the ML lifecycle. Which one applies to a tabular credit-scoring model, and which one applies to a Bedrock RAG bot, and why is mixing them up the single most-cited trap question on the MLA exam? If a stakeholder asks you for the AWS tool that filters unsafe LLM prompts, which one do you reach for, and what would happen if you reached for the other?

## Exam-Style Practice

**Question 1 (Domain 1, pre-training bias metric selection):**

A team is auditing a credit-scoring training set for pre-training bias on the `gender` facet. The set has 5000 rows; 80% of approved applications come from one facet group and 20% from the other. Which Clarify pre-training bias metric directly captures this proportion-of-labels imbalance?

A. Difference in Proportions of Labels (DPL).

B. Disparate Impact (DI).

C. Conditional Demographic Disparity (CDD).

D. Specificity Difference (SD).

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: DPL measures the difference in proportions of the favored label across the protected facet groups. For the scenario (80/20 split of approvals across two gender groups), DPL is the direct measure: it equals `P(favored | facet_a) - P(favored | facet_b)` = 0.8 - 0.2 = 0.6, well above the 0.1 threshold.
- B is wrong: Disparate Impact is a post-training metric (requires model predictions), not pre-training. It compares the rate of favored outcomes between facet groups in the model's predictions, not in the labels themselves.
- C is wrong: Conditional Demographic Disparity conditions the proportion of labels on a subgroup feature (for example, DPL within each age band). It is a refinement on top of DPL, not the primary metric for the scenario.
- D is wrong: Specificity Difference is a post-training metric comparing true-negative rates across facets; it is not a pre-training label-imbalance metric.

</details>

**Question 2 (Domain 4, Clarify vs Bedrock Guardrails distinction):**

A team is launching a Bedrock-powered customer-support RAG bot AND retraining a tabular fraud-scoring model. Both teams hear "AWS has a tool for bias and safety." Which combination of tools fits each use case?

A. Use SageMaker Clarify for the RAG bot to filter unsafe prompts and the fraud-scoring model to measure statistical fairness.

B. Use Bedrock Guardrails for the RAG bot to filter unsafe prompts and PII leaks, and SageMaker Clarify for the fraud-scoring model to measure pre-training and post-training statistical fairness.

C. Use Bedrock Guardrails for both because Guardrails is the unified AWS safety service.

D. Use SageMaker Model Monitor for both because Model Monitor handles all production safety.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: Clarify does NOT filter prompts. Clarify is a statistical fairness and explainability tool for tabular ML models. It does not understand text or LLM prompts.
- B is correct: Bedrock Guardrails is the content-safety tool for foundation models and Bedrock-powered apps (denied topics, PII redaction, prompt-injection filters). Clarify is the statistical-fairness tool for tabular ML training (pre-training and post-training bias metrics, explainability via SHAP). They solve different problems on different parts of the ML lifecycle.
- C is wrong: Bedrock Guardrails does not measure statistical fairness on a tabular model. It works on text prompts and responses through Bedrock APIs.
- D is wrong: Model Monitor monitors PRODUCTION inference for data and model quality drift, not pre-training bias and not LLM content safety.

</details>